In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'SACOG Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Housing', 'config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Housing', 'config')

path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')
path_out     = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Production')


In [ ]:
# TODO:
# Read in Region_SACOG_Permit_Data.xlsx file by year concatenated together into one dataset
# Then organize the data as described in the crosswalk.xlsx file
# For this script, let's do the indicators:
## Production_1, Production_4, Production_6, Location_1, Location_2a, Location_2b, and Policy_5

In [ ]:
## Fun tip to comment multiple lines of code at the same time:
# Highlight all lines of code
# Hold ctrl + ?


In [ ]:
year_start = 2001
year_end   = 2022

years_to_import = range(year_start, year_end+1)

list_df = []

for year in years_to_import:
    df_year = pd.read_excel(os.path.join(path_housing, 'Region_SACOG_Permit_Data.xlsx'), sheet_name = str(year))
    df_year['Year'] = year
    list_df.append(df_year)

df_housing = pd.concat(list_df)
df_housing = df_housing.set_index(['County', 'Jurisdiction', 'Year']).reset_index()
df_housing.loc[df_housing['Jurisdiction'].str.contains('COUNTY'), 'Jurisdiction'] = 'UNINCORPORATED'
df_housing

***

## Production_1

***

In [ ]:
indicator_name = 'Production_1'

print('Organizing indicator Production_1 by Jurisdictions')
df_prod1_a = df_housing.copy()
df_prod1_a = df_pop1_a[df_pop1_a['County'] != 'Region']
df_prod1_a = df_prod1_a[['County', 'Jurisdiction', 'Year', 'Total']]
display(df_prod1_a.head(5))

print('Organizing indicator Production_1 by Counties')
df_prod1_b = df_housing.copy()
df_prod1_b = df_prod1_b[df_prod1_b['County'] != 'Region']
df_prod1_b = df_prod1_b.groupby(['County', 'Year'], as_index = False)['Total'].agg('sum')
display(df_prod1_b.head(5))

print('Organizing indicator Production_1 by the entire SACOG Region')
df_prod1_c = df_housing.copy()
df_prod1_c = df_prod1_c[df_prod1_c['County'] != 'Region']
df_prod1_c = df_prod1_c.groupby(['Year'], as_index = False)['Total'].agg('sum')
display(df_prod1_c.head(5))

# Export
df_prod1_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + '_SACOG Housing Permit Data.xlsx'), index = False)
df_prod1_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + '_SACOG Housing Permit Data.xlsx'), index = False)
df_prod1_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + '_SACOG Housing Permit Data.xlsx'), index = False)

***

## Production_4

***

In [ ]:

# Use pd.read_excel to import the Pop_5 jurisdiction data
# Calculate population growth by year using df_pop_5
# Calculate housing growth by year using df_housing "Total" column
# Merge the two files together by county/jurisdiction
# Roll up to Jurisdiction, County, and MPO levels (3 different data frames)
# Make plots

df_pop = pd.read_excel('DF_pop_5_jurisdiction')


Code graveyard

In [ ]:
# df_housing_2001 = pd.read_excel(os.path.join(path_housing, 'Region_SACOG_Permit_Data.xlsx'), sheet_name = '2001')
# df_housing_2002 = pd.read_excel(os.path.join(path_housing, 'Region_SACOG_Permit_Data.xlsx'), sheet_name = '2002')
# df_housing_2003 = pd.read_excel(os.path.join(path_housing, 'Region_SACOG_Permit_Data.xlsx'), sheet_name = '2003')
# df_housing_2004 = pd.read_excel(os.path.join(path_housing, 'Region_SACOG_Permit_Data.xlsx'), sheet_name = '2004')

# df_housing_2001['Year'] = 2001
# df_housing_2002['Year'] = 2002
# df_housing_2003['Year'] = 2003
# df_housing_2004['Year'] = 2004

# list_df = [df_housing_2001, df_housing_2002, df_housing_2003, df_housing_2004]
# df_housing = pd.concat(list_df)
# df_housing = df_housing.set_index(['County', 'Jurisdiction', 'Year']).reset_index()
# df_housing